In [2]:
### This code is for utilizing the data from HIVIS
# HIVIS: https://apps.usgs.gov/hivis
# url generation for API: https://waterservices.usgs.gov/test-tools/
#    -> select instantanious values service

# given a list of usgs site codes and a time frame this code will create spreadsheets of the sites with hydrological information

##########################################################################################################################################
import requests
import pandas as pd
from functools import reduce

#USGS site codes (get from HIVIS)
site_ids = [
    '02146409',    #lil sugar creek, Charlotte, NC
    '01462000',    #Delaware River at Lambertville NJ
]

startDT = '2025-12-31T00:01'
endDT = '2026-01-31T23:59'

def make_spreadsheet(site_id, startDT, endDT, out_file='usgs_data.xlsx'):
    # get json from USGS API
    base_url = "https://waterservices.usgs.gov/nwis/iv/"
    params = {
        "format": 'json',
        "sites": site_id,
        "startDT": startDT,
        "endDT": endDT,
        "siteStatus": 'all',
    }
    r = requests.get(base_url, params=params)
    r.raise_for_status()
    data = r.json()
    
    time_series = data['value']['timeSeries']
    
    dfs = []
    for series in time_series:
        site_name = series["sourceInfo"]["siteName"]
        site_code = series["sourceInfo"]["siteCode"][0]["value"]
        variable = series["variable"]["variableName"]
        unit = series["variable"]["unit"]["unitCode"]
    
        values = series["values"][0]["value"]
        if not values:
            continue
    
        df = pd.DataFrame(values)
        df["dateTime"] = pd.to_datetime(df["dateTime"]).dt.tz_localize(None)
        df["value"] = pd.to_numeric(df["value"], errors="coerce")
    
        # Rename value column to variable name
        col_name = f"{variable} ({unit})"
        df = df[["dateTime", "value"]].rename(columns={"value": col_name})
    
        dfs.append(df)

    # Combine into one dataframe
    if not dfs:
        print(f"No data for site {site_id}")
        return
    
    full_df = reduce(
        lambda left, right: pd.merge(left, right, on="dateTime", how="outer"),
        dfs
    )

    full_df["siteName"] = site_name
    full_df["siteCode"] = site_code

    full_df = full_df.sort_values("dateTime")

    # Save to Excel
    full_df.to_excel(out_file, index=False)

for site_id in site_ids:
    make_spreadsheet(site_id, startDT, endDT, out_file='usgs_data_'+site_id+'.xlsx')
    
    
    